In [2]:
import requests
import pandas as pd
import pycountry


In [3]:
def fetch_country_crude_production(
    api_key,
    start="2000-01",
    end=None
):
    url = "https://api.eia.gov/v2/international/data/"

    params = {
        "api_key": api_key,
        "frequency": "monthly",
        "data[0]": "value",

        # Crude oil including lease condensate
        "facets[productId][]": "57",

        # Production
        "facets[activityId][]": "1",

        # Thousand barrels per day
        "facets[unit][]": "TBPD",

        "start": start,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": 5000,
    }

    if end:
        params["end"] = end

    all_rows = []
    offset = 0

    while True:
        params["offset"] = offset

        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()

        data = response.json()["response"]
        rows = data["data"]

        if not rows:
            break

        all_rows.extend(rows)
        offset += len(rows)

        print(f"Fetched {offset}/{data['total']} rows", end="\r")

        if offset >= int(data["total"]):
            break

    df = pd.DataFrame(all_rows)

    df["date"] = pd.to_datetime(df["period"])
    df["production_tbpd"] = pd.to_numeric(df["value"], errors="coerce")

    columns = [
        "date",
        "countryRegionId",
        "countryRegionName",
        "production_tbpd",
    ]
    df = df[[col for col in columns if col in df.columns]]

    df = df.rename(
        columns={
            "countryRegionId": "country_code",
            "countryRegionName": "country",
        }
    )

    # IMPORTANT:
    # EIA International also returns World and regional aggregates
    # (Africa, Western Europe, etc.). Keep only real ISO-3166 countries.
    valid_country_codes = {country.alpha_3 for country in pycountry.countries}

    df = df[
        df["country_code"].isin(valid_country_codes)
    ].copy()

    df = (
        df.dropna(subset=["date", "production_tbpd"])
          .drop_duplicates(subset=["date", "country_code"], keep="last")
          .sort_values(["date", "country"])
          .reset_index(drop=True)
    )

    return df


In [5]:
API_KEY = "kMWwLaP6bX0Qwf2zdKR4cE4hVt4r2cDQjpRZJeCg"

country_production_df = fetch_country_crude_production(
    api_key=API_KEY,
    start="2000-01"
)

country_production_df.head()


,date,country_code,country,production_tbpd
0,2000-01-01,ALB,Albania,5.716
1,2000-01-01,DZA,Algeria,800.000
2,2000-01-01,AGO,Angola,718.000
3,2000-01-01,ARG,Argentina,815.497
4,2000-01-01,AUS,Australia,756.880


In [6]:
# Current OPEC members
OPEC_CODES = {
    "DZA",  # Algeria
    "COG",  # Republic of the Congo
    "GNQ",  # Equatorial Guinea
    "GAB",  # Gabon
    "IRN",  # Iran
    "IRQ",  # Iraq
    "KWT",  # Kuwait
    "LBY",  # Libya
    "NGA",  # Nigeria
    "SAU",  # Saudi Arabia
    "ARE",  # United Arab Emirates
    "VEN",  # Venezuela
}

# Keep only non-OPEC countries BEFORE pivoting/summing
non_opec_country_df = country_production_df[
    ~country_production_df["country_code"].isin(OPEC_CODES)
].copy()

# One country per column
non_opec_production_wide = (
    non_opec_country_df.pivot_table(
        index="date",
        columns="country",
        values="production_tbpd",
        aggfunc="sum",
    )
    .reset_index()
)

# Restrict to the analysis period
non_opec_production = non_opec_production_wide[
    (non_opec_production_wide["date"] >= "2006-09-01") &
    (non_opec_production_wide["date"] <= "2026-05-31")
].copy()

non_opec_production["month"] = (
    non_opec_production["date"].dt.strftime("%Y-%m")
)

# Sum ONLY country production columns.
country_columns = [
    col for col in non_opec_production.columns
    if col not in ["date", "month"]
]

non_opec_production["total_non_opec_production_tbpd"] = (
    non_opec_production[country_columns]
    .sum(axis=1, min_count=1)
)

# TBPD -> million barrels per day
# 1,000 thousand barrels/day = 1 million barrels/day
non_opec_production["total_non_opec_production_mbpd"] = (
    non_opec_production["total_non_opec_production_tbpd"] / 1000
)

# Optional: actual monthly volume in million barrels/month
non_opec_production["total_non_opec_production_million_bbl_month"] = (
    non_opec_production["total_non_opec_production_tbpd"]
    * non_opec_production["date"].dt.days_in_month
    / 1000
)

# Quick checks: these should both be False
print("World included:", "World" in non_opec_production.columns)
print(
    "Any OPEC country included:",
    non_opec_country_df["country_code"].isin(OPEC_CODES).any()
)

non_opec_production[
    [
        "month",
        "total_non_opec_production_tbpd",
        "total_non_opec_production_mbpd",
        "total_non_opec_production_million_bbl_month",
    ]
].head()


World included: False
Any OPEC country included: False


country,month,total_non_opec_production_tbpd,total_non_opec_production_mbpd,total_non_opec_production_million_bbl_month
80,2006-09,44297.514,44.297514,1328.925420
81,2006-10,44736.178,44.736178,1386.821518
82,2006-11,44872.222,44.872222,1346.166660
83,2006-12,44786.917,44.786917,1388.394427
84,2007-01,45125.933,45.125933,1398.903923


In [7]:

non_opec_production.to_csv("non_opec_production.csv",  index= False)